# Spark Session Configuration

Reusable Spark session configuration for Microsoft Fabric notebooks.
Import this notebook via `%run nb_spark_config` at the top of any pipeline-triggered
notebook to standardize session settings across your workspace.

**Why `spark.conf.set()` instead of `SparkSession.builder`?**
Fabric pre-initializes the Spark session. `SparkSession.builder.getOrCreate()` returns
the existing session, and some configs (especially Delta-level settings) may not take
effect when passed through the builder after session creation. Using `spark.conf.set()`
applies settings to the already-running session reliably.

**Note:** Some settings (like `spark.sql.shuffle.partitions`) can be changed at any point.
Others (like `spark.sql.caseSensitive`) affect query planning and should be set before
any DataFrames are created in the session.

In [ ]:
# ── Configuration Flags ─────────────────────────────────────────────────────────
# Toggle these booleans and values to control session behavior.
# When imported via %run, the calling notebook inherits these settings.

case_sensitive = True          # Treat column names as case-sensitive (important for mixed-case source schemas)
shuffle_partitions = 64        # Default is 200 — reduce for small-to-mid datasets to avoid excessive empty partitions
adaptive_query_execution = True  # AQE dynamically adjusts shuffle partitions, join strategies, and skew handling at runtime
adaptive_skew_join = True      # Splits skewed partitions during joins to prevent stragglers (requires AQE enabled)
fast_optimize = True           # Fabric-specific: enables bin-packing optimization during OPTIMIZE for faster compaction
schema_evolution = True        # Auto-merge schema changes on write (new columns added automatically)
one_security = False           # OneLake security enforcement — disable when not using workspace-level security policies

In [ ]:
# ── Apply Configuration ─────────────────────────────────────────────────────────
# Uses spark.conf.set() against the pre-initialized Fabric Spark session.

# Case sensitivity — must be set before creating any DataFrames
spark.conf.set("spark.sql.caseSensitive", case_sensitive)

# Shuffle + Adaptive Query Execution
spark.conf.set("spark.sql.shuffle.partitions", shuffle_partitions)
spark.conf.set("spark.sql.adaptive.enabled", adaptive_query_execution)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", adaptive_skew_join)

# Delta / Fabric-specific
spark.conf.set("spark.microsoft.delta.optimize.fast.enabled", fast_optimize)
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", schema_evolution)

# OneLake Security
spark.conf.set("spark.onelake.security.enabled", one_security)

In [ ]:
# ── Verify Configuration ────────────────────────────────────────────────────────
# Print active settings for confirmation. Safe to remove in production.

config_keys = [
    ("Case sensitive",           "spark.sql.caseSensitive"),
    ("Shuffle partitions",       "spark.sql.shuffle.partitions"),
    ("Adaptive query execution", "spark.sql.adaptive.enabled"),
    ("Adaptive skew join",       "spark.sql.adaptive.skewJoin.enabled"),
    ("Fast optimize",            "spark.microsoft.delta.optimize.fast.enabled"),
    ("Schema evolution",         "spark.databricks.delta.schema.autoMerge.enabled"),
    ("OneSecurity",              "spark.onelake.security.enabled"),
]

print("--- Spark Configuration ---")
for label, key in config_keys:
    print(f"  {label + ':':<30} {spark.conf.get(key)}")
print("--- Configuration applied ---")